# Q2 风险分位数策略最终版

最终预测模型已经冻结：

$$
W=60,\quad K_L=3,\quad K_P=2,
$$

$$
h_L=lag1,\qquad h_P=lag1+lag7+mean7,
$$

$$
C_L=weekday7,\qquad C_P=annual1,
$$

$$
\alpha_L=0.3,\qquad \alpha_P=10.
$$

最近 28 天成对残差形成净负荷经验场景。

风险分位数只在 **2025-05-01 至 2025-08-31 开发期**扫描

$$
q\in\{0.05,0.10,\ldots,0.95\},
$$

以实际总费用最小为目标确定全局实施参数 $q^*$。Sep–Dec 的预测评价不参与 $q^*$ 选择。


In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from q2_final import run_all

report = run_all(
    ROOT/"input"/"q2_actual_load_pv.csv",
    ROOT/"input"/"q2_fixed_prices.csv",
    ROOT/"rerun_results",
)
report


In [ ]:
scan = pd.read_csv(ROOT/"rerun_results"/"qstar_development_scan.csv")
display(scan.sort_values("total_cost_yuan").round(4))


开发期经验最优为

$$
\boxed{q^*=0.85}.
$$

单时段五倍应急损失给出的理论参考为

$$
q_0=1-\frac{1}{5}=0.80.
$$

$q^*=0.85$ 略高于单时段理论值，反映了储能的跨时段耦合、效率损失与分时电价共同作用。


In [ ]:
daily = pd.read_csv(ROOT/"rerun_results"/"final_strategy_daily.csv")
daily[["planned_grid_kwh","emergency_kwh","planned_cost_yuan","emergency_cost_yuan","total_cost_yuan"]].sum()


## 因果口径

每日预测特征和残差场景只使用决策日之前已经结束的真实日数据。

$q^*$ 以及预测超参数属于**离线全局校准参数**，固定后用于完整策略回放；它们不会在某一天的在线决策中读取当天未来真实负荷或光伏。
